# Authentication with Keycloak

This notebook demonstrates how to authenticate with the ADEPT ecosystem using Keycloak.

## Security Architecture

JupyterLab is isolated on `frontend_net` — all authentication requests route through
`nginx_proxy` which proxies to Keycloak on the application network. Tokens acquired this
way carry `https://nginx_proxy/realms/master` as the issuer, which is accepted by all
ADEPT services.

## Authentication Methods
1. **Resource Owner Password Credentials (ROPC)** - For end users
2. **Client Credentials** - For service-to-service communication
3. **JWT Token Management** - Token refresh and validation

## Prerequisites
- Completed `00_environment_setup.ipynb`
- ADEPT services running (`make start`)

## 1. Setup and Imports

In [ ]:
import os
import sys
import json
import httpx
import asyncio
from datetime import datetime, timedelta
from typing import Optional, Dict, Any
import jwt
from getpass import getpass

# Load configuration from setup notebook
try:
    with open("/tmp/adept_config.json", "r") as f:
        config = json.load(f)
        NGINX_URL = config["nginx_url"]
        API_BASE_URL = config["api_base_url"]
        KEYCLOAK_TOKEN_URL = config["keycloak_token_url"]
        print("Configuration loaded from setup notebook")
except FileNotFoundError:
    print("Please run 00_environment_setup.ipynb first")
    # Fallback: route through nginx_proxy (network-isolated default)
    NGINX_URL = "https://nginx_proxy"
    API_BASE_URL = f"{NGINX_URL}/v1"
    KEYCLOAK_TOKEN_URL = f"{NGINX_URL}/realms/master/protocol/openid-connect/token"

KEYCLOAK_USERINFO_URL = f"{NGINX_URL}/realms/master/protocol/openid-connect/userinfo"
print(f"Token endpoint: {KEYCLOAK_TOKEN_URL}")
print(f"API base URL:   {API_BASE_URL}")

## 2. User Authentication (ROPC Flow)

In [ ]:
class KeycloakAuthenticator:
    """Helper class for Keycloak authentication via nginx_proxy."""
    
    def __init__(self, base_url: str, realm: str = "master"):
        self.base_url = base_url
        self.realm = realm
        self.token_url = f"{base_url}/realms/{realm}/protocol/openid-connect/token"
        self.userinfo_url = f"{base_url}/realms/{realm}/protocol/openid-connect/userinfo"
        self.access_token = None
        self.refresh_token = None
        self.token_expiry = None
    
    async def authenticate_user(self, username: str, password: str, client_id: str = "login-client") -> Dict:
        """Authenticate a user using ROPC flow via nginx_proxy."""
        async with httpx.AsyncClient(verify=False, timeout=10.0) as client:
            response = await client.post(
                self.token_url,
                data={
                    "grant_type": "password",
                    "client_id": client_id,
                    "username": username,
                    "password": password
                }
            )
            
            if response.status_code == 200:
                token_data = response.json()
                self.access_token = token_data["access_token"]
                self.refresh_token = token_data.get("refresh_token")
                self.token_expiry = datetime.now() + timedelta(seconds=token_data.get("expires_in", 300))
                return {
                    "success": True,
                    "access_token": self.access_token,
                    "expires_in": token_data.get("expires_in"),
                    "token_type": token_data.get("token_type")
                }
            else:
                return {
                    "success": False,
                    "error": response.text,
                    "status_code": response.status_code
                }
    
    async def get_user_info(self) -> Dict:
        """Get information about the authenticated user."""
        if not self.access_token:
            return {"error": "Not authenticated"}
        
        async with httpx.AsyncClient(verify=False, timeout=10.0) as client:
            response = await client.get(
                self.userinfo_url,
                headers={"Authorization": f"Bearer {self.access_token}"}
            )
            
            if response.status_code == 200:
                return response.json()
            else:
                return {"error": response.text}
    
    def decode_token(self) -> Dict:
        """Decode JWT token to see claims (no verification, inspection only)."""
        if not self.access_token:
            return {"error": "No token available"}
        
        try:
            decoded = jwt.decode(self.access_token, options={"verify_signature": False})
            return decoded
        except Exception as e:
            return {"error": str(e)}

# Create authenticator — routes through nginx_proxy
auth = KeycloakAuthenticator(NGINX_URL, "master")

## 3. Authenticate as Notebook User

In [ ]:
# Default credentials for notebook user
username = "notebook-user"
password = os.getenv("NOTEBOOK_USER_PASSWORD", "notebook-password")

print(f"Authenticating as: {username}")
result = await auth.authenticate_user(username, password)

if result["success"]:
    print("Authentication successful!")
    print(f"Token type: {result['token_type']}")
    print(f"Expires in: {result['expires_in']} seconds")
    print(f"\nAccess token (first 50 chars): {result['access_token'][:50]}...")
else:
    print(f"Authentication failed: {result['error']}")
    print("\nTroubleshooting:")
    print("1. Verify services are running: make start")
    print("2. Verify nginx_proxy is healthy: docker logs nginx_proxy --tail 5")
    print("3. Rebuild JupyterLab if config changed: make rebuild-jupyterlab")

## 4. Inspect Token Claims

In [4]:
# Decode and inspect the JWT token
if auth.access_token:
    claims = auth.decode_token()
    
    if "error" not in claims:
        print("JWT Token Claims:")
        print("-" * 50)
        
        # Important claims
        print(f"Subject (user ID): {claims.get('sub')}")
        print(f"Username: {claims.get('preferred_username')}")
        print(f"Email: {claims.get('email')}")
        print(f"Groups: {claims.get('groups', [])}")
        print(f"Issued at: {datetime.fromtimestamp(claims.get('iat', 0))}")
        print(f"Expires at: {datetime.fromtimestamp(claims.get('exp', 0))}")
        
        # Roles
        realm_roles = claims.get('realm_access', {}).get('roles', [])
        print(f"\nRealm roles: {realm_roles}")
        
        # Resource access
        resource_access = claims.get('resource_access', {})
        if resource_access:
            print("\nResource access:")
            for resource, access in resource_access.items():
                print(f"  {resource}: {access.get('roles', [])}")
    else:
        print(f"Error decoding token: {claims['error']}")

JWT Token Claims:
--------------------------------------------------
Subject (user ID): e03b7004-0dda-423d-9fd4-482632cb9f4e
Username: notebook-user
Email: None
Groups: ['notebook-users']
Issued at: 2025-11-07 21:17:48
Expires at: 2025-11-08 05:17:48

Realm roles: ['default-roles-master', 'offline_access', 'uma_authorization']

Resource access:
  account: ['manage-account', 'manage-account-links', 'view-profile']


## 5. Get User Information

In [5]:
# Get detailed user information from Keycloak
user_info = await auth.get_user_info()

if "error" not in user_info:
    print("User Information:")
    print("-" * 50)
    for key, value in user_info.items():
        print(f"{key}: {value}")
else:
    print(f"Error getting user info: {user_info['error']}")

Error getting user info: 


## 6. Test Authenticated API Call

In [ ]:
# Test an authenticated call through nginx_proxy
async def test_authenticated_api():
    """Test API call with authentication token via nginx_proxy."""
    if not auth.access_token:
        print("Not authenticated. Please run authentication cell first.")
        return
    
    headers = {
        "Authorization": f"Bearer {auth.access_token}",
        "Content-Type": "application/json"
    }
    
    async with httpx.AsyncClient(verify=False, timeout=10.0) as client:
        # Test health endpoint (no auth required)
        print("Testing health endpoint (no auth required):")
        response = await client.get(f"{NGINX_URL}/health")
        print(f"  Status: {response.status_code}")
        
        # Test authenticated endpoint
        print("\nTesting authenticated endpoint (/v1/models):")
        response = await client.get(
            f"{API_BASE_URL}/models",
            headers=headers
        )
        print(f"  Status: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            print(f"  Available models: {len(data.get('data', []))}")
            for model in data.get('data', [])[:3]:
                print(f"    - {model.get('id')}")
        elif response.status_code in (401, 403):
            print("  Authentication failed - token may be invalid or expired")
        else:
            print(f"  Response: {response.text[:200]}")

await test_authenticated_api()

## 7. Service Account Authentication

In [ ]:
async def authenticate_service_account(client_id: str, client_secret: str) -> Dict:
    """Authenticate using client credentials (service account) via nginx_proxy."""
    async with httpx.AsyncClient(verify=False, timeout=10.0) as client:
        response = await client.post(
            KEYCLOAK_TOKEN_URL,
            data={
                "grant_type": "client_credentials",
                "client_id": client_id,
                "client_secret": client_secret
            }
        )
        
        if response.status_code == 200:
            token_data = response.json()
            print(f"Service account '{client_id}' authenticated")
            return {
                "success": True,
                "access_token": token_data["access_token"],
                "expires_in": token_data["expires_in"]
            }
        else:
            print(f"Authentication failed: {response.text}")
            return {"success": False, "error": response.text}

# Example: Authenticate as validation-runner-client
# client_secret = os.getenv("VALIDATION_RUNNER_CLIENT_SECRET", "")
# service_auth = await authenticate_service_account("validation-runner-client", client_secret)

print("Service account authentication available for:")
print("  - validation-runner-client (Validation tests)")
print("  - agent-gateway-client (Agent Gateway service)")
print("  - gateway-registry-client (Registry service)")
print("  - openwebui-client (OpenWebUI integration)")
print("  - n8n-client (n8n workflow automation)")

## 8. Token Refresh

In [ ]:
async def refresh_access_token(refresh_token: str, client_id: str = "login-client") -> Dict:
    """Refresh an access token using a refresh token via nginx_proxy."""
    async with httpx.AsyncClient(verify=False, timeout=10.0) as client:
        response = await client.post(
            KEYCLOAK_TOKEN_URL,
            data={
                "grant_type": "refresh_token",
                "client_id": client_id,
                "refresh_token": refresh_token
            }
        )
        
        if response.status_code == 200:
            return {"success": True, "data": response.json()}
        else:
            return {"success": False, "error": response.text}

# Test token refresh if we have a refresh token
if auth.refresh_token:
    print("Testing token refresh...")
    refresh_result = await refresh_access_token(auth.refresh_token)
    
    if refresh_result["success"]:
        print("Token refreshed successfully")
        new_token = refresh_result["data"]["access_token"]
        print(f"New token (first 50 chars): {new_token[:50]}...")
    else:
        print(f"Refresh failed: {refresh_result['error']}")
else:
    print("No refresh token available")

## 9. Save Authentication for Other Notebooks

In [ ]:
# Save authentication details for use in other notebooks
if auth.access_token:
    auth_config = {
        "access_token": auth.access_token,
        "refresh_token": auth.refresh_token,
        "token_expiry": auth.token_expiry.isoformat() if auth.token_expiry else None,
        "username": username,
        "nginx_url": NGINX_URL,
        "api_base_url": API_BASE_URL,
        "authenticated_at": datetime.now().isoformat()
    }
    
    auth_file = "/tmp/adept_auth.json"
    with open(auth_file, "w") as f:
        json.dump(auth_config, f, indent=2)
    
    print(f"Authentication saved to {auth_file}")
    print("   This token can be used in other notebooks")
    print(f"   API access via: {API_BASE_URL}")
else:
    print("No authentication token to save")

## 10. Summary and Next Steps

In [ ]:
print("=" * 60)
print("AUTHENTICATION SETUP COMPLETE")
print("=" * 60)

if auth.access_token:
    print("Successfully authenticated")
    print(f"   User: {username}")
    print(f"   Routing: All requests via nginx_proxy (TLS + JWT)")
    
    # Check token expiry
    if auth.token_expiry:
        time_left = auth.token_expiry - datetime.now()
        print(f"   Token expires in: {time_left.total_seconds():.0f} seconds")
    
    # Decode and show issuer
    claims = auth.decode_token()
    if "error" not in claims:
        print(f"   Issuer: {claims.get('iss')}")
    
    print("\nNext Steps:")
    print("\nChoose your learning path:")
    print("\n1. END USER Journey (../01-end-user/):")
    print("   - 01_chat_interface.ipynb - Interactive chat with agents")
    print("   - 02_file_operations.ipynb - Upload and process files")
    print("   - 03_scientific_workflows.ipynb - Run scientific analyses")
    
    print("\n2. DEVELOPER Guide (../02-developer/):")
    print("   - 01_sdk_basics.ipynb - Using the ADEPT SDK")
    print("   - 02_building_mcp_tools.ipynb - Create custom tools")
    print("   - 03_custom_agents.ipynb - Build custom agents")
    
    print("\n3. OPERATOR Manual (../03-operator/):")
    print("   - 01_deployment.ipynb - Deploy the ecosystem")
    print("   - 02_monitoring.ipynb - Monitor services")
    print("   - 03_scaling.ipynb - Scale the platform")
else:
    print("Authentication failed")
    print("\nTroubleshooting:")
    print("1. Verify services: make start")
    print("2. Verify nginx_proxy: docker logs nginx_proxy --tail 10")
    print("3. Rebuild JupyterLab: make rebuild-jupyterlab")
    print("4. Check docker-compose config:")
    print("   deployment/local/docker-compose/docker-compose.core.yaml")
    print("   (jupyterlab service must have ADEPT_API_BASE_URL=https://nginx_proxy/v1)")